# 5주차 실습: 텍스트 표현 방법

**주제:** Bag-of-Words(BoW), TF-IDF, n-gram 특징 추출

## 학습 목표
1. 토큰화된 텍스트를 수치 벡터로 표현해야 하는 이유를 설명한다.
2. CountVectorizer로 BoW 행렬을 만들고 희소행렬의 의미를 해석한다.
3. n-gram의 장점과 차원 증가·희소성의 한계를 판단한다.
4. TF-IDF의 TF·DF·IDF를 구분하고 과업에 맞게 설정한다.

> **제출:** 모든 TODO를 작성·실행하고 체크포인트 답변을 Markdown 셀에 남기세요. 외부 AI·자료를 사용했다면 사용 범위와 검증 과정을 기록하세요.

## 0. 준비
교육용 예시 데이터입니다. 실제 데이터는 출처·라이선스·개인정보(PII) 포함 여부를 확인하세요.

In [2]:
# 오류가 날 때만 아래 주석을 해제하세요.
!pip -q install -U scikit-learn pandas kiwipiepy

import pandas as pd
from kiwipiepy import Kiwi
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

kiwi = Kiwi()
corpus = [
    '배송이 빠르고 제품 품질이 좋아요',
    '배송은 느렸지만 제품 디자인은 예뻐요',
    '품질은 꽤 좋지만 가격이 조금 비싸요', # '꽤' 추가. 하지만 tokenizer가 지정되지 않은 경우, '(?u)\\b\\w\\w+\\b' 정규표현식으로 처리되기 때문에 한글자짜리 한글은 제거됨(RECEIVED HELP FROM AI)
    '가격 대비 품질이 만족스러워요',
    '배송이 너무 느리고 포장도 아쉬워요'
]
df = pd.DataFrame({'doc_id': range(1, len(corpus)+1), 'text': corpus}) # doc_id 열과 text 열 만들고, text 열의 값으로 corpus 지정
display(df)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 MB 9.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 28.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.


,doc_id,text
0,1,배송이 빠르고 제품 품질이 좋아요
1,2,배송은 느렸지만 제품 디자인은 예뻐요
2,3,품질은 꽤 좋지만 가격이 조금 비싸요
3,4,가격 대비 품질이 만족스러워요
4,5,배송이 너무 느리고 포장도 아쉬워요


## 1. 텍스트를 벡터로 표현하는 이유
머신러닝 모델은 문장 자체가 아니라 숫자 입력을 계산합니다. 4주차의 형태소·토큰을 열(feature)로 만들고, 문서마다 각 토큰의 값(빈도 또는 중요도)을 채웁니다.

`원문 → 정제·토큰화 → 문서-토큰 행렬 → 분류/예측`

In [3]:
# TODO 1: 공백 기준 토큰화로 CountVectorizer를 만들고 문서-토큰 행렬을 확인하세요.
count_vec = CountVectorizer() # CountVectorizer 객체 만듦. 말뭉치를 어휘 사전으로 구축하고, 단어 출현 빈도로 수치화
X_count = count_vec.fit_transform(corpus) # 인덱스 생성(fit)과 벡터화(transform) 함께 진행(인덱스 순서는 한글 사전식). 숫자 벡터 행렬 반환.

print(count_vec.get_feature_names_out()) # 전체 단어장 확인
display(pd.DataFrame(X_count.toarray(), columns=count_vec.get_feature_names_out())) # 0이 상당히 많음. '은, 는, 이, 가'가 붙으면서 별도의 단어가 되는 것도 영향을 미침(ex. 가격, 가격이)

['가격' '가격이' '너무' '느렸지만' '느리고' '대비' '디자인은' '만족스러워요' '배송은' '배송이' '비싸요' '빠르고'
 '아쉬워요' '예뻐요' '제품' '조금' '좋아요' '좋지만' '포장도' '품질은' '품질이']


,가격,가격이,너무,느렸지만,느리고,대비,디자인은,만족스러워요,배송은,배송이,...,빠르고,아쉬워요,예뻐요,제품,조금,좋아요,좋지만,포장도,품질은,품질이
0,0,0,0,0,0,0,0,0,0,1,...,1,0,0,1,0,1,0,0,0,1
1,0,0,0,1,0,0,1,0,1,0,...,0,0,1,1,0,0,0,0,0,0
2,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,1,0,1,0
3,1,0,0,0,0,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,1
4,0,0,1,0,1,0,0,0,0,1,...,0,1,0,0,0,0,0,1,0,0


### 체크포인트 1
문서-토큰 행렬에서 **행**, **열**, **값**이 각각 무엇을 뜻하는지 1문장씩 설명하세요. 행렬의 0이 많은 이유도 설명하세요.

- 행은 모든 단어의 집합이고, 열은 특정 단어입니다. 값은 해당 문장에서 특정 단어의 빈도수를 나타냅니다.
- 실제 말뭉치는 수만 개 이상의 단어로 구성됩니다. 이에 따라 전체 단어 사전은 고차원 벡터인데, 각 문서에는 일부 단어만 포함되기에 문서 벡터의 안 채워진 단어들에 대해서는(대부분) 원소가 0으로 채워집니다(희소 행렬).

- scikit-learn은 메모리 낭비를 방지하기 위해, 희소행렬을 저장할 때 0이 아닌 값과 그 인덱스만 저장하는 CSR(Compressed Sparse Row) 형식을 기본으로 사용합니다. 이 방식을 통해 메모리 효율성을 높이고, 연산 속도를 어느 정도 높일 수 있습니다. __*(AI)*__

## 2. Bag-of-Words(BoW)
BoW는 각 단어의 등장 횟수를 사용합니다. 구현은 간단하지만 단어 순서와 긴 문맥 정보를 잃습니다. `배송이 빠르다`와 `빠르지 않은 배송`처럼 순서·부정이 중요한 경우를 생각해 보세요.

In [4]:
# TODO 2: max_features=5를 적용한 BoW를 만들고, 어떤 단어가 남았는지 확인하세요.
small_count_vec = CountVectorizer(max_features=5) # 빈도수 기준 상위 5개의 특성만 체크하는 CountVectorizer
X_small = small_count_vec.fit_transform(corpus) # 단어장 인덱스 생성 및 벡터화
print(small_count_vec.get_feature_names_out()) # 가격에 대한 정보가 없음. 배송에 대해 긍정 내용이 그냥 빠져버림.

['가격' '느리고' '배송이' '제품' '품질이']


- 차원의 저주라고 해서, 학습 시 과적합 위험이 있습니다. CSR로 저장해도, 차원의 크기는 그대로이니 설명을 위한 데이터(문제)가 많이 필요하고 부족 시 과적합 위험이 생기는 구조입니다. __*(AI)*__

## 3. n-gram: 인접한 토큰 묶기
- unigram: `배송`, `빠르다`
- bigram: `배송 빠르다`
- trigram: `배송 이 빠르다`

n-gram은 일부 어순·표현 정보를 보완하지만, n이 커질수록 특징 수가 늘고 희소성이 커질 수 있습니다.

In [5]:
# TODO 3: unigram과 bigram을 함께 쓰는 CountVectorizer를 만드세요.
ngram_vec = CountVectorizer(ngram_range=(1, 2)) # uni, bi.
# 둘을 같이 쓰는 이유: bigram으로 부정표현, 고유개념 등 최소한의 순서 정보를 포착할 수 있으면서도 이에 따른 차원 크기 증가는 감수 가능한 수준이기 때문
# unigram은 구체적인 bigram이 등장하지 않아도 unigram이 등장해 최소한의 정보는 보존해주는 역할(+개별 키워드 남김). --> RECEIVED HELP FROM AI

X_ngram = ngram_vec.fit_transform(corpus)
print(ngram_vec.get_feature_names_out())
print('특징 수:', X_ngram.shape[1]) # '너무'와 '너무 느리고'가 동시에 있는 등, 특징의 수는 늘어남(21->40). '가격'과 '조금'이 붙어 부정의 의미를 유추할 수 있음.

['가격' '가격 대비' '가격이' '가격이 조금' '너무' '너무 느리고' '느렸지만' '느렸지만 제품' '느리고'
 '느리고 포장도' '대비' '대비 품질이' '디자인은' '디자인은 예뻐요' '만족스러워요' '배송은' '배송은 느렸지만' '배송이'
 '배송이 너무' '배송이 빠르고' '비싸요' '빠르고' '빠르고 제품' '아쉬워요' '예뻐요' '제품' '제품 디자인은'
 '제품 품질이' '조금' '조금 비싸요' '좋아요' '좋지만' '좋지만 가격이' '포장도' '포장도 아쉬워요' '품질은'
 '품질은 좋지만' '품질이' '품질이 만족스러워요' '품질이 좋아요']
특징 수: 40


### 체크포인트 2
감성 분석에서 bigram이 unigram보다 유용할 수 있는 표현을 하나 만들고, 그 이유를 설명하세요. 특징 수 증가가 항상 좋은 것은 아닌 이유도 함께 적으세요.

- '안 좋다', '기계 학습'이 대표적인 예시입니다. bigram으로 둘을 붙여서 인식함에 따라 부정 표현이라는 점, 그리고 고유 개념이라는 점을 포착할 수 있기 때문입니다.
- 하지만 특징 수 증가가 항상 좋지만은 않습니다. 그 이유는 조합이 많이 될수록, 자주 나올 일이 떨어져 데이터 부족으로 인한 과적합 위험이 생기기 때문입니다.__*(AI)*__
- 또한, LM(언어 모델) 관점에서는 조합이 길다면 훈련 데이터 내에 정확히 일치하는 예문이 존재할 확률이 낮기 때문에, 자연스러운 문장임에도 자연스러운 문장일 확률을 0%에 가깝게 판단하는 문제가 있습니다.__*(AI)*__

- 단순 빈도(BoW)는 '은, 는, 이, 가' 등 문법적으로 많이 쓰이는 단어에 너무 높은 중요도를 부여해, 실제 단어가 묻히는 한계가 있습니다.

## 4. TF-IDF: 많이 나오지만 덜 구별되는 단어의 가중치 낮추기
- **TF (Term Frequency):** 한 문서 안에서의 단어 빈도 (특정 문서 안에서의 단어의 상대적 중요도).
- **DF (Document Frequency):** 해당 단어를 포함한 문서 수 (전체 말뭉치에 걸쳐 얼마나 널리 분포하는지). DF가 높다면 모든 문서에 흔하게 등장하는 것이므로, 정보 가치는 낮다고 볼 수 있음.
- **IDF (Inverse Document Frequency):** 여러 문서에 흔한 단어의 가중치를 낮추는 값

TF-IDF는 단어의 중요도를 표현하는 한 방법이며, 과업에 따라 BoW가 더 적절할 수도 있습니다.

In [6]:
# TODO 4: TfidfVectorizer로 TF-IDF 행렬을 만들고 소수점 둘째 자리까지 확인하세요.
tfidf_vec = TfidfVectorizer() # TfidfVectorizer 만듦.
X_tfidf = tfidf_vec.fit_transform(corpus) # 단어장 인덱스 생성 및 벡터화(내부 연산: TF와 IDF의 곱)
tfidf_df = pd.DataFrame(X_tfidf.toarray(), columns=tfidf_vec.get_feature_names_out()) # 행렬로 변환
display(tfidf_df.round(2)) # 소수점 둘째 자리까지 확인

,가격,가격이,너무,느렸지만,느리고,대비,디자인은,만족스러워요,배송은,배송이,...,빠르고,아쉬워요,예뻐요,제품,조금,좋아요,좋지만,포장도,품질은,품질이
0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.41,...,0.5,0.00,0.00,0.41,0.00,0.5,0.00,0.00,0.00,0.41
1,0.00,0.00,0.00,0.46,0.00,0.00,0.46,0.00,0.46,0.00,...,0.0,0.00,0.46,0.37,0.00,0.0,0.00,0.00,0.00,0.00
2,0.00,0.45,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.0,0.00,0.00,0.00,0.45,0.0,0.45,0.00,0.45,0.00
3,0.52,0.00,0.00,0.00,0.00,0.52,0.00,0.52,0.00,0.00,...,0.0,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.42
4,0.00,0.00,0.46,0.00,0.46,0.00,0.00,0.00,0.00,0.37,...,0.0,0.46,0.00,0.00,0.00,0.0,0.00,0.46,0.00,0.00


In [7]:
# TODO 5: 아래 문서에서 BoW와 TF-IDF의 상위 3개 토큰을 비교하세요.
target_idx = 1 # 1번 문장('배송은 느렸지만 제품 디자인은 예뻐요')에 대해서
count_scores = pd.Series(X_count.toarray()[target_idx], index=count_vec.get_feature_names_out()) # BoW로 얻은 벡터
tfidf_scores = pd.Series(X_tfidf.toarray()[target_idx], index=tfidf_vec.get_feature_names_out()) # Tfidf로 얻은 벡터
print('BoW 상위:', count_scores.sort_values(ascending=False).head(3)) # 내림차순으로 정렬한 뒤 위에서 3개 = 상위 3개
print('TF-IDF 상위:', tfidf_scores.sort_values(ascending=False).head(3)) # 가중치가 낮아짐(현재는 예시에 나온 빈도수가 많아야 1번이라 수치가 좀 그런데, 데이터 많아지면 더 다양해짐)

BoW 상위: 느렸지만    1
예뻐요     1
제품      1
dtype: int64
TF-IDF 상위: 느렸지만    0.463693
배송은     0.463693
예뻐요     0.463693
dtype: float64


- CountVectorizer는 특정 단어가 많이 나오는 문서 집계 시 유리합니다.
  - 스팸 메일 -> '광고'라는 단어 도배(광고라는 단어 자체는 전체 문서에서 희소하지 않아 tfidf는 가중치 줄여버림)__*(AI)*__
  - 특정 단어가 많이 등장하는 문서는 몇개인지, 몇번이나 언급되는지 등의 직관적인 통계 데이터 뽑을 때.__*(AI)*__
- Tfidfvectorizer는 특정 단어가 텍스트의 핵심 의미를 얼마나 잘 대표하는지가 중요한 과제에 적합합니다.
  - 문서 간 유사도 분석(표절 검사, 중복 문서 제거)__*(AI)*__
  - 텍스트 분류(뉴스 기사 분류, 감성 분석)__*(AI)*__

### 체크포인트 3
여러 문서에 공통으로 등장하는 단어는 TF-IDF에서 일반적으로 어떤 영향을 받나요? 이 단어가 과업상 중요할 수도 있는 사례를 하나 드세요.

- 중요도가 크게 낮아집니다.
- '슬프다'와 같은 단어는 감정 분석에서 중요할텐데, 모든 문서에서 반복되다보면(ex. 슬픈 영화 리뷰) tfidf 상에서는 중요도가 낮아질 가능성이 있습니다. 이처럼 tfidf는 해당 데이터셋 전체의 기저 톤이 되는 단어를 무시할 수 있습니다.__*(AI)*__
- 이를 해결하기 위한 방법으로는 감정 사전을 기반으로 가중치를 높게 매기거나, 단순 빈도수만 세거나, 임베딩 모델을 활용해 전체의 부정적인 문맥과 뉘앙스를 파악하거나, 참조 코퍼스 활용을 통한 감정 단어의 중요성 보존(참조 코퍼스로 사전학습 + 내 코퍼스로 추론) 등이 있습니다.__*(AI)*__

## 5. 한국어 형태소 토큰을 벡터화하기
공백 토큰화 대신 4주차의 형태소 분석 결과를 벡터화할 수 있습니다. 여기서는 키워드 추출 예시로 명사 계열을 사용합니다. 감성 분석이라면 용언·부사·부정 표현 보존을 추가로 검토해야 합니다.

In [11]:
def kiwi_noun_tokens(text):
    return [token.form for token in kiwi.tokenize(text)
            if token.tag.startswith('N') and len(token.form) > 1] # 명사 계열이고, 단어가 두 글자 이상이면 리스트에 추가해서, 해당 리스트 반환

print('예시 토큰:', kiwi_noun_tokens(corpus[0]))

# TODO 6: tokenizer에 kiwi_noun_tokens를 연결한 TfidfVectorizer를 만들고 결과를 확인하세요.
korean_tfidf = TfidfVectorizer(tokenizer=kiwi_noun_tokens, token_pattern=None)
X_korean = korean_tfidf.fit_transform(corpus)
print(korean_tfidf.get_feature_names_out())
display(pd.DataFrame(X_korean.toarray(), columns=korean_tfidf.get_feature_names_out()).round(2))

예시 토큰: ['배송', '제품', '품질']
['가격' '대비' '디자인' '만족' '배송' '제품' '포장' '품질']


,가격,대비,디자인,만족,배송,제품,포장,품질
0,0.00,0.00,0.00,0.00,0.54,0.65,0.00,0.54
1,0.00,0.00,0.69,0.00,0.46,0.56,0.00,0.00
2,0.77,0.00,0.00,0.00,0.00,0.00,0.00,0.64
3,0.46,0.57,0.00,0.57,0.00,0.00,0.00,0.38
4,0.00,0.00,0.00,0.00,0.56,0.00,0.83,0.00


## 6. 미니 과제: 특징 추출 정책 설계
다음 중 하나를 선택하세요. (1) 강의평 키워드 추출 (2) 상품 리뷰 감성 분석 (3) 학교 민원 자동 분류

분석기/토큰 단위, 품사 선택, n-gram 범위, 최소 문서 빈도(`min_df`), 사용할 표현(BoW 또는 TF-IDF)을 근거와 함께 결정하세요.

In [9]:
# TODO 7: 선택한 과업에 맞게 아래 설정을 수정하고 실행하세요.
# 상품 리뷰 감성 분석을 진행합니다.

mini_corpus = [ # 출처: https://shopping.naver.com/logistics/products/11586892597?NaPm=ct%3Dmv4mpqy3%7Cci%3Dshoppingvertical%7Ctr%3Davgth%7Chk%3Dee1f0923c8fe3a0e1e2f9a225d2738066c9fbc00#REVIEW_DIALOG
    "배송빠르고 좋지만 맛이없네요", # 1점
    "가성비 완전 꽝! 맛도 사과 맛 만 조금나고, 이 가격이면 이보다 더 좋은것 먹을것 같음.", # 1점
    "처음 구매해서 먹어봅니다. 유명해서 먹어벌려고 구매했어요. 담백하고 맛과 향이 좋아요.", # 5점
    "일단 한 박스당 펼치기 쉽게 패키징되어 있어서 좋았어요! 그런데 대두 함량이 커서 그런지 애플파이 맛이 나긴 해도 뭔가 두 입만 먹어도 물렸던 것 같아요..! 약간 실망스럽긴 하지만 그래도 칼로리나 다른 부분은 만족합니더!", # 4점
    "맛이 다양해서 질리지 않게 먹을 수 있어요." # 5점
]
mini_df = pd.DataFrame({"doc_id": range(1, len(mini_corpus)+1), "text": mini_corpus})
display(mini_df)

# sol1. 공백 기준 토큰화 -> 문서-토큰 행렬 확인
mini_vec = CountVectorizer(token_pattern=r"(?u)\b\w+\b") # default(한글자는 제거) 대신 실제 공백 기준 토큰화 적용(유니코드 활성화한 상태에서 단어 하나와 일치시킴). RECEIVED HELP FROM AI
X_mini = mini_vec.fit_transform(mini_corpus)
print(mini_vec.get_feature_names_out())
display(pd.DataFrame(X_mini.toarray(), columns = mini_vec.get_feature_names_out()))

,doc_id,text
0,1,배송빠르고 좋지만 맛이없네요
1,2,"가성비 완전 꽝! 맛도 사과 맛 만 조금나고, 이 가격이면 이보다 더 좋은것 먹을것..."
2,3,처음 구매해서 먹어봅니다. 유명해서 먹어벌려고 구매했어요. 담백하고 맛과 향이 좋아요.
3,4,일단 한 박스당 펼치기 쉽게 패키징되어 있어서 좋았어요! 그런데 대두 함량이 커서 ...
4,5,맛이 다양해서 질리지 않게 먹을 수 있어요.


['가격이면' '가성비' '같아요' '같음' '것' '구매해서' '구매했어요' '그래도' '그런데' '그런지' '꽝' '나긴'
 '다른' '다양해서' '담백하고' '대두' '더' '두' '만' '만족합니더' '맛' '맛과' '맛도' '맛이' '맛이없네요'
 '먹어도' '먹어벌려고' '먹어봅니다' '먹을' '먹을것' '물렸던' '뭔가' '박스당' '배송빠르고' '부분은' '사과' '수'
 '쉽게' '실망스럽긴' '않게' '애플파이' '약간' '완전' '유명해서' '이' '이보다' '일단' '입만' '있어서' '있어요'
 '조금나고' '좋아요' '좋았어요' '좋은것' '좋지만' '질리지' '처음' '칼로리나' '커서' '패키징되어' '펼치기'
 '하지만' '한' '함량이' '해도' '향이']


,가격이면,가성비,같아요,같음,것,구매해서,구매했어요,그래도,그런데,그런지,...,처음,칼로리나,커서,패키징되어,펼치기,하지만,한,함량이,해도,향이
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,1,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,1,1,0,0,0,...,1,0,0,0,0,0,0,0,0,1
3,0,0,1,0,1,0,0,1,1,1,...,0,1,1,1,1,1,1,1,1,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [14]:
# sol2. 형태소 기반 토큰화 + tfidf
def kiwi_verb_tokens(text):
  return [token.form for token in kiwi.tokenize(text)
          if token.tag.startswith('V') and len(token.form) >= 1] # 형용사나 동사 계열이고 한글자 이상이면 리스트에 추가해서 리스트 반환

my_vectorizer = TfidfVectorizer(
    tokenizer=kiwi_verb_tokens,
    token_pattern=None,
    ngram_range=(1, 1),
    min_df=1
)

X_my = my_vectorizer.fit_transform(mini_corpus)
print('특징 수:', X_my.shape[1])
print('특징:', my_vectorizer.get_feature_names_out())

display(pd.DataFrame(X_my.toarray(), columns = my_vectorizer.get_feature_names_out()))

특징 수: 19
특징: ['같' '그렇' '나' '담백하' '먹' '물리' '벌' '보' '빠르' '쉽' '않' '없' '이' '있' '좋' '질리' '크'
 '펼치' '하']


,같,그렇,나,담백하,먹,물리,벌,보,빠르,쉽,않,없,이,있,좋,질리,크,펼치,하
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.6569,0.000000,0.000000,0.6569,0.000000,0.000000,0.370086,0.000000,0.000000,0.000000,0.000000
1,0.501555,0.000000,0.501555,0.000000,0.350235,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.0000,0.501555,0.000000,0.350235,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.000000,0.466913,0.526101,0.000000,0.466913,0.466913,0.0000,0.000000,0.000000,0.0000,0.000000,0.000000,0.263050,0.000000,0.000000,0.000000,0.000000
3,0.230621,0.285849,0.230621,0.000000,0.161042,0.285849,0.000000,0.000000,0.0000,0.285849,0.000000,0.0000,0.230621,0.230621,0.161042,0.000000,0.285849,0.285849,0.571698
4,0.000000,0.000000,0.000000,0.000000,0.327000,0.000000,0.000000,0.000000,0.0000,0.000000,0.580423,0.0000,0.000000,0.468282,0.000000,0.580423,0.000000,0.000000,0.000000


### 체크포인트 4
선택한 과업의 특징 추출 정책을 설명하세요. 다음을 모두 포함합니다.
1. BoW/TF-IDF 선택 이유  2. 토큰·품사 선택  3. n-gram과 min_df 설정  4. 예상 오류 또는 정보 손실 사례
- 한국어 감성 분석은 형태소 분석이 필요하므로, 분석기는 kiwi를 사용하여 형태소 분석을 진행했습니다.
- 감성 분석에 필요한 동사와 형용사를 선택하였습니다.
- ngram의 범위는 (1, 2)로 할 경우 다른 위치에 있는 동사나 형용사끼리 붙어 순서에 혼선을 줄 여지가 있기에, (1,1)로 했습니다.
- 데이터셋 구성 시 긍정과 부정을 절반씩 썼기 때문에, 전체가 긍정이나 부정에 치우쳐 tfidf의 문제 발생은 없다고 생각해서, 성능이 좋은 tfidf를 선정했습니다.

## 제출 체크리스트
- [x] TODO 1~7을 모두 작성하고 실행했다.
- [x] BoW 행렬의 행·열·값·희소성을 설명했다.
- [x] unigram과 bigram의 차이·한계를 설명했다.
- [x] TF·DF·IDF의 역할을 설명했다.
- [x] 공백 기반과 형태소 기반 벡터화를 비교했다.
- [x] 선택 과업에 맞는 특징 추출 정책과 오류 사례를 기록했다.
- [x] 외부 AI/자료 사용 시 사용 범위와 검증 과정을 기록했다.

## 다음 주차 예고
다음 시간에는 만들어진 문서-토큰 행렬을 입력으로 사용하여 나이브 베이즈와 로지스틱 회귀 기반 텍스트 분류 모델을 학습하고 평가합니다.

## 형성평가
- Question 01: IDF가 높은 단어일수록 전체 코퍼스 내에서 자주 등장하는 흔한 단어이다? (X/O)
  - X
- Question 02: BoW의 가장 큰 한계점임 '단어 순서 정보 유실'을 보완하기 위해 사용하는 기법은?
  - ngram